In [5]:
import json
import numpy as np

class Website:
    def __init__(self, id, content, links, category):
        self.id = id
        self.content = content
        self.links = links
        self.category = category

def load_web(file):
    # load websites data from JSON
    with open(file, 'r') as f:
        data = json.load(f)
        website_data = data.get('websites', [])  # accessing 'websites' key
    return [Website(**website) for website in website_data]


def cal_adjacency_matrix(websites):
    # adjacency matrix for websites based on their links
    n = len(websites)
    adjacency_matrix = np.zeros((n, n))
    id_to_index = {website.id: index for index, website in enumerate(websites)}
    for i, website in enumerate(websites):
        for link in website.links:
            if link in id_to_index:
                j = id_to_index[link]
                adjacency_matrix[i, j] = 1
    return adjacency_matrix

# hubs and authorities algorithm
def hubs_and_authorities(adjacency_matrix, num_iter=100):
    n = adjacency_matrix.shape[0]
    auth_score = np.ones(n)
    hub_score = np.ones(n)
    for _ in range(num_iter):
        auth_score = adjacency_matrix.T @ hub_score
        hub_score = adjacency_matrix @ auth_score
        auth_score /= np.linalg.norm(auth_score)
        hub_score /= np.linalg.norm(hub_score)
    return auth_score, hub_score

def search(websites, adjacency_matrix, word):
    # search for relevant websites containing the taken word
    auth_score, hub_score = hubs_and_authorities(adjacency_matrix)
    relevant_websites = [(website, hub_score[websites.index(website)], auth_score[websites.index(website)]) for website in websites if word in website.content]
    relevant_websites.sort(key=lambda x: x[2], reverse=True)
    return relevant_websites

def main():
    websites = load_web('/content/drive/MyDrive/DATASET.json')
    adj_matrix = cal_adjacency_matrix(websites)
    word = input('Enter the word you wanna search: ')
    relevant_websites = search(websites, adj_matrix, word)

    # calculating and displaying results
    # for a neat table
    max_id_length = max(len(str(website.id)) for website, _, _ in relevant_websites)
    max_category_length = max(len(website.category) for website, _, _ in relevant_websites)
    max_lengths = [len("Rank"), len("ID"), len("Category"), len("Hub Score"), len("Authority Score")]

    lengths = [max(max_lengths[0], len(str(len(relevant_websites)))),
                      max(max_id_length, max_lengths[1]),
                      max(max_category_length, max_lengths[2]),
                      max_lengths[3],
                      max_lengths[4]]

    print(f'\nNumber of results for the word "{word}" = {len(relevant_websites)}')

    top_k = int(input('How many top sites do you want to be displayed? '))
    top_websites = relevant_websites[:top_k]

    print(f'\nTop {len(top_websites)} websites for the word "{word}" are:\n')
    line = '|' + '-' * (lengths[0] + 2) + '|' + '-' * (lengths[1] + 2) + '|' + '-' * (lengths[2] + 2) + '|' + '-' * 12 + '|' + '-' * 19 + '|'
    print(line)
    print(f'| {"Rank":<{lengths[0]}} | {"ID":<{lengths[1]}} | {"Category":<{lengths[2]}} | {"Hub Score":^10} | {"Authority Score":^17} |')
    print(line)

    for idx, (website, hub_score, auth_score) in enumerate(top_websites, start=1):
        print(f'| {idx:<{lengths[0]}} | {website.id:<{lengths[1]}} | {website.category:<{lengths[2]}} | {hub_score:^10.5f} | {auth_score:^17.5f} |')

if __name__ == '__main__':
    main()

Enter the word you wanna search: race

Number of results for the word "race" = 19
How many top sites do you want to be displayed? 10

Top 10 websites for the word "race" are:

|------|-----|----------|------------|-------------------|
| Rank | ID  | Category | Hub Score  |  Authority Score  |
|------|-----|----------|------------|-------------------|
| 1    | 557 | news     |  0.01150   |      0.03611      |
| 2    | 519 | sport    |  0.02468   |      0.03497      |
| 3    | 585 | sport    |  0.04563   |      0.03400      |
| 4    | 395 | sport    |  0.03377   |      0.03332      |
| 5    | 481 | news     |  0.01271   |      0.03325      |
| 6    | 81  | fun      |  0.03657   |      0.03303      |
| 7    | 289 | fun      |  0.04372   |      0.03196      |
| 8    | 720 | fun      |  0.03646   |      0.03180      |
| 9    | 957 | sport    |  0.02207   |      0.03175      |
| 10   | 27  | news     |  0.02337   |      0.03168      |
